# Tu bucket en MinIO — operaciones básicas (CRUD)

Cada uno tiene **un bucket personal** en MinIO para sus laboratorios: `jhunior`, `luis`,
`pablo`, `martin` o `manuel`. Este cuaderno enseña lo que se hace con él:

| | Operación | Sección |
|---|---|---|
| **C** | subir texto, archivos, tablas, imágenes | 2 |
| **R** | listar, leer, descargar, ver metadatos | 3 |
| **U** | sobrescribir, renombrar, mover | 4 |
| **D** | borrar un objeto o una carpeta entera | 5 |

**Reglas del bucket personal**

- Es **tuyo y privado**: nadie más lo lee ni escribe (ni tú el de los demás).
- **No se respalda.** Es espacio para pruebas; lo que importe, a git o al proyecto.
- **No hay que configurar nada**: tu cuaderno arranca ya con tu credencial.

Ejecuta las celdas **en orden**. Todo lo que crea este cuaderno va bajo el prefijo
`demo-crud/` de tu bucket, y la sección 7 lo limpia.

## 1 · Conexión

In [ ]:
import os, io, json
import boto3, botocore
import pandas as pd
import numpy as np
from PIL import Image

# El endpoint y tu credencial ya están en el entorno del cuaderno.
s3 = boto3.client("s3", 
                  endpoint_url=os.environ["AWS_ENDPOINT_URL_S3"],
                  aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                  endpoint_url=os.environ["AWS_SECRET_ACCESS_KEY"],
                 )

# Tu identidad solo ve los buckets a los que tiene acceso: los dos del proyecto y el tuyo.
visibles = [b["Name"] for b in s3.list_buckets()["Buckets"]]
propios = [b for b in visibles if b not in {"datasets", "modelos"}]
assert len(propios) == 1, f"Esperaba un bucket personal y veo {propios} (visibles: {visibles})"

BUCKET = propios[0]
PREFIJO = "demo-crud/"          # todo lo de este cuaderno cuelga de aquí
print("Tu bucket:", BUCKET, "| visibles:", visibles)

> Si la celda falla con `KeyError: 'AWS_ENDPOINT_URL_S3'` o con el `assert`, tu servidor
> arrancó antes de que existiera tu credencial: **File → Hub Control Panel → Stop My Server**
> y vuelve a arrancarlo.

**Dos ideas antes de empezar:**

- En S3 **no hay carpetas**. Hay *claves* (`demo-crud/datos/tabla.csv`) y la `/` es un
  carácter más; las «carpetas» que ves en la consola son solo prefijos comunes.
- Un objeto **no se edita**: se reemplaza entero. Por eso *Update* es escribir otra vez.

## 2 · Create — subir cosas

In [ ]:
# 2.1 · Texto o bytes, directamente desde memoria
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "notas/hola.txt",
              Body="Hola desde mi cuaderno\n".encode("utf-8"),
              ContentType="text/plain; charset=utf-8")

# 2.2 · Un diccionario como JSON
config = {"experimento": "prueba-1", "lr": 0.001, "epocas": 10}
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "config/prueba-1.json",
              Body=json.dumps(config, ensure_ascii=False, indent=2).encode("utf-8"),
              ContentType="application/json")

# 2.3 · Un archivo del disco de tu cuaderno (upload_file parte los grandes solo)
with open("ejemplo_local.txt", "w") as f:
    f.write("línea 1\nlínea 2\n")
s3.upload_file("ejemplo_local.txt", BUCKET, PREFIJO + "archivos/ejemplo_local.txt")

# 2.4 · Una tabla de pandas como CSV, sin pasar por disco
df = pd.DataFrame({"especie": ["Solanum", "Puya", "Polylepis"],
                   "altitud_m": [3200, 4100, 3900]})
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "datos/tabla.csv",
              Body=df.to_csv(index=False).encode("utf-8"), ContentType="text/csv")

# 2.5 · Una imagen generada en memoria, como PNG
img = Image.fromarray((np.random.rand(64, 64, 3) * 255).astype("uint8"))
buf = io.BytesIO(); img.save(buf, format="PNG")
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "imagenes/ruido.png",
              Body=buf.getvalue(), ContentType="image/png")

# 2.6 · Metadatos propios: pares clave-valor que viajan con el objeto
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "notas/con-metadatos.txt", Body=b"x",
              Metadata={"autor": "yo", "curso": "vision"})

print("Subidos.")

## 3 · Read — listar, leer, descargar

In [ ]:
# 3.1 · Listar. SIEMPRE con paginador: una llamada suelta devuelve como mucho 1 000
#       objetos, y contar sobre ella hace creer que faltan datos.
def listar(prefijo="", bucket=BUCKET):
    filas = []
    for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefijo):
        for o in pagina.get("Contents", []):
            filas.append({"clave": o["Key"], "bytes": o["Size"],
                          "modificado": o["LastModified"].strftime("%Y-%m-%d %H:%M")})
    return pd.DataFrame(filas, columns=["clave", "bytes", "modificado"])

listar(PREFIJO)

In [ ]:
# 3.2 · Ver solo el primer nivel, como carpetas (Delimiter="/")
r = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIJO, Delimiter="/")
[p["Prefix"] for p in r.get("CommonPrefixes", [])]

In [ ]:
# 3.3 · Leer a memoria: texto, JSON, CSV e imagen
texto  = s3.get_object(Bucket=BUCKET, Key=PREFIJO + "notas/hola.txt")["Body"].read().decode("utf-8")
config = json.loads(s3.get_object(Bucket=BUCKET, Key=PREFIJO + "config/prueba-1.json")["Body"].read())
tabla  = pd.read_csv(s3.get_object(Bucket=BUCKET, Key=PREFIJO + "datos/tabla.csv")["Body"])
imagen = Image.open(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=PREFIJO + "imagenes/ruido.png")["Body"].read()))

print(repr(texto)); print(config); print(imagen.size)
tabla

In [ ]:
# 3.4 · Descargar a un archivo del cuaderno
s3.download_file(BUCKET, PREFIJO + "datos/tabla.csv", "tabla_descargada.csv")
open("tabla_descargada.csv").read()

In [ ]:
# 3.5 · Metadatos sin bajar el contenido (head_object)
h = s3.head_object(Bucket=BUCKET, Key=PREFIJO + "notas/con-metadatos.txt")
{"bytes": h["ContentLength"], "tipo": h["ContentType"], "etag": h["ETag"], "propios": h["Metadata"]}

In [ ]:
# 3.6 · ¿Existe un objeto? head_object da 404 si no — distinguirlo de otros errores
def existe(clave, bucket=BUCKET):
    try:
        s3.head_object(Bucket=bucket, Key=clave)
        return True
    except botocore.exceptions.ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise                       # 403 u otro: no es «no existe», es «no puedo mirar»

existe(PREFIJO + "notas/hola.txt"), existe(PREFIJO + "no/esta.txt")

## 4 · Update — sobrescribir, renombrar, mover

No hay «editar»: se vuelve a escribir la misma clave, o se copia a una nueva.

In [ ]:
# 4.1 · Modificar = leer, cambiar, volver a escribir la MISMA clave
tabla = pd.read_csv(s3.get_object(Bucket=BUCKET, Key=PREFIJO + "datos/tabla.csv")["Body"])
tabla.loc[len(tabla)] = ["Gentianella", 4500]
s3.put_object(Bucket=BUCKET, Key=PREFIJO + "datos/tabla.csv",
              Body=tabla.to_csv(index=False).encode("utf-8"), ContentType="text/csv")

pd.read_csv(s3.get_object(Bucket=BUCKET, Key=PREFIJO + "datos/tabla.csv")["Body"])

In [ ]:
# 4.2 · Añadir líneas a un texto: igual, no hay «append» en S3
clave = PREFIJO + "notas/hola.txt"
viejo = s3.get_object(Bucket=BUCKET, Key=clave)["Body"].read().decode("utf-8")
s3.put_object(Bucket=BUCKET, Key=clave, Body=(viejo + "otra línea\n").encode("utf-8"),
              ContentType="text/plain; charset=utf-8")
s3.get_object(Bucket=BUCKET, Key=clave)["Body"].read().decode("utf-8")

In [ ]:
# 4.3 · Renombrar o mover = copiar a la clave nueva + borrar la vieja
def mover(origen, destino, bucket=BUCKET):
    s3.copy_object(Bucket=bucket, Key=destino, CopySource={"Bucket": bucket, "Key": origen})
    s3.delete_object(Bucket=bucket, Key=origen)

mover(PREFIJO + "archivos/ejemplo_local.txt", PREFIJO + "archivos/renombrado.txt")
listar(PREFIJO + "archivos/")

In [ ]:
# 4.4 · Cambiar solo los metadatos: copiar el objeto sobre sí mismo con REPLACE
clave = PREFIJO + "notas/con-metadatos.txt"
s3.copy_object(Bucket=BUCKET, Key=clave, CopySource={"Bucket": BUCKET, "Key": clave},
               Metadata={"autor": "yo", "curso": "nlp", "revisado": "si"},
               MetadataDirective="REPLACE")
s3.head_object(Bucket=BUCKET, Key=clave)["Metadata"]

## 5 · Delete — borrar

In [ ]:
# 5.1 · Un objeto. OJO: borrar una clave que no existe TAMBIÉN responde bien (S3 no avisa)
s3.delete_object(Bucket=BUCKET, Key=PREFIJO + "imagenes/ruido.png")
existe(PREFIJO + "imagenes/ruido.png")      # False

In [ ]:
# 5.2 · Todo un prefijo («carpeta»). Primero en seco: enseña qué borraría, no borra nada.
def borrar_prefijo(prefijo, bucket=BUCKET, de_verdad=False):
    assert prefijo and prefijo.endswith("/"), "Prefijo vacío o sin '/' final: demasiado peligroso"
    claves = listar(prefijo, bucket)["clave"].tolist()
    if not de_verdad:
        print(f"[en seco] borraría {len(claves)} objetos bajo {bucket}/{prefijo}")
        return claves
    for i in range(0, len(claves), 1000):              # delete_objects admite 1 000 por lote
        lote = [{"Key": k} for k in claves[i:i + 1000]]
        r = s3.delete_objects(Bucket=bucket, Delete={"Objects": lote, "Quiet": True})
        if r.get("Errors"):
            raise RuntimeError(r["Errors"])
    print(f"borrados {len(claves)} objetos bajo {bucket}/{prefijo}")
    return claves

borrar_prefijo(PREFIJO + "config/")                    # en seco

In [ ]:
borrar_prefijo(PREFIJO + "config/", de_verdad=True)
listar(PREFIJO)

## 6 · Lo que NO puedes hacer (y cómo se ve)

Conviene reconocer el error cuando aparezca: es `AccessDenied`, no un fallo del cuaderno.

In [ ]:
def intentar(descripcion, funcion):
    try:
        funcion()
        print(f"✅ {descripcion}: permitido")
    except botocore.exceptions.ClientError as e:
        print(f"⛔ {descripcion}: {e.response['Error']['Code']}")

ajeno = next(b for b in ["jhunior", "luis", "pablo", "martin", "manuel"] if b != BUCKET)

intentar(f"leer el bucket de otra persona ({ajeno})",
         lambda: s3.list_objects_v2(Bucket=ajeno, MaxKeys=1))
intentar("leer el dataset del proyecto",
         lambda: s3.list_objects_v2(Bucket="datasets", Prefix="herbario/piloto/", MaxKeys=1))
intentar("escribir dentro del piloto del proyecto",
         lambda: s3.put_object(Bucket="datasets", Key="herbario/piloto/_no.txt", Body=b"x"))
intentar("crear un bucket nuevo",
         lambda: s3.create_bucket(Bucket=f"{BUCKET}-extra"))

Esperado: el bucket ajeno, escribir en el piloto y crear buckets dan `AccessDenied`; leer el
dataset del proyecto está permitido. Si necesitas algo de eso, pídelo al operador.

## 7 · Limpieza

In [ ]:
borrar_prefijo(PREFIJO, de_verdad=True)
for f in ("ejemplo_local.txt", "tabla_descargada.csv"):
    if os.path.exists(f):
        os.remove(f)
listar(PREFIJO)          # vacío

## Chuleta

```python
s3.put_object(Bucket=BUCKET, Key="a/b.txt", Body=b"...")        # crear / sobrescribir
s3.upload_file("local.csv", BUCKET, "a/local.csv")               # subir archivo
s3.get_object(Bucket=BUCKET, Key="a/b.txt")["Body"].read()       # leer
s3.download_file(BUCKET, "a/local.csv", "copia.csv")             # descargar
listar("a/")                                                      # listar (con paginación)
s3.head_object(Bucket=BUCKET, Key="a/b.txt")                     # metadatos
s3.copy_object(Bucket=BUCKET, Key="c.txt", CopySource={"Bucket": BUCKET, "Key": "a/b.txt"})
s3.delete_object(Bucket=BUCKET, Key="a/b.txt")                   # borrar
```

Para mirar tus archivos sin código: **`consola-s3.ai-unmsm.com`**, entrando con GitHub.